In [13]:
import requests
import pandas as pd
import time

# proxy city coordinates
cities = {
    'National':      (28.61, 77.21),  # Delhi as stand-in
    'Northern':      (28.61, 77.21),  # Delhi
    'Western':       (19.08, 72.88),  # Mumbai
    'Eastern':       (22.57, 88.36),  # Kolkata
    'Southern':      (13.08, 80.27),  # Chennai
    'North-Eastern': (26.14, 91.74),  # Guwahati
}

def fetch_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()['hourly']
    return pd.DataFrame({
        'datetime': pd.to_datetime(data['time']),
        'temperature': data['temperature_2m'],
        'humidity': data['relative_humidity_2m']
    })

weather_frames = []
for location, (lat, lon) in cities.items():
    print(f"Fetching {location}...")
    df = fetch_weather(lat, lon, "2019-01-01", "2024-04-30")
    df['location'] = location
    weather_frames.append(df)
    time.sleep(1)  # be polite to the free API, avoid rate limiting

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df.to_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/weather.parquet', index=False)
weather_df.head()

Fetching National...
Fetching Northern...
Fetching Western...
Fetching Eastern...
Fetching Southern...
Fetching North-Eastern...


,datetime,temperature,humidity,location
0,2019-01-01 00:00:00,8.8,84,National
1,2019-01-01 01:00:00,8.1,86,National
2,2019-01-01 02:00:00,7.6,87,National
3,2019-01-01 03:00:00,7.3,88,National
4,2019-01-01 04:00:00,7.1,88,National


In [14]:
import pandas as pd

load_long = pd.read_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/load_long.parquet')
load_long.head()  # sanity check it loaded correctly

,datetime,location,demand_gw
0,2019-01-01 00:00:00,National,118.69067
1,2019-01-01 01:00:00,National,116.02923
2,2019-01-01 02:00:00,National,114.04414
3,2019-01-01 03:00:00,National,113.64897
4,2019-01-01 04:00:00,National,116.29005


In [15]:
merged = load_long.merge(weather_df, on=['datetime', 'location'], how='left')
print(merged.isna().sum())  # check nothing dropped silently
merged.to_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/merged.parquet', index=False)
merged.head()

datetime       0
location       0
demand_gw      0
temperature    0
humidity       0
dtype: int64


,datetime,location,demand_gw,temperature,humidity
0,2019-01-01 00:00:00,National,118.69067,8.8,84
1,2019-01-01 01:00:00,National,116.02923,8.1,86
2,2019-01-01 02:00:00,National,114.04414,7.6,87
3,2019-01-01 03:00:00,National,113.64897,7.3,88
4,2019-01-01 04:00:00,National,116.29005,7.1,88


In [16]:
print(sorted(load_long['location'].unique()))
print(sorted(weather_df['location'].unique()))

['Eastern', 'National', 'North-Eastern', 'Northern', 'Southern', 'Western']
['Eastern', 'National', 'North-Eastern', 'Northern', 'Southern', 'Western']


In [17]:
load_long['location'] = (
    load_long['location']
    .str.replace(' Region', '', regex=False)
    .str.strip()
    .replace({'Northen': 'Northern'})
)

print(sorted(load_long['location'].unique()))
# should now print: ['Eastern', 'National', 'North-Eastern', 'Northern', 'Southern', 'Western']

['Eastern', 'National', 'North-Eastern', 'Northern', 'Southern', 'Western']


In [18]:
load_long.to_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/load_long.parquet', index=False)
merged = load_long.merge(weather_df, on=['datetime', 'location'], how='left')
print(merged.isna().sum())  # should now be all zeros

datetime       0
location       0
demand_gw      0
temperature    0
humidity       0
dtype: int64


In [22]:
import holidays

# time-based features — trivial from datetime
merged['hour'] = merged['datetime'].dt.hour
merged['day_of_week'] = merged['datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
merged['month'] = merged['datetime'].dt.month

# holiday flag
india_holidays = holidays.India(years=range(2019, 2025))
merged['is_holiday'] = merged['datetime'].dt.normalize().isin(
    pd.to_datetime(list(india_holidays.keys()))
).astype(int)

# historical average load — the average demand for this location at this hour/day-of-week combo
# (this becomes a feature: "what does demand at 2pm on a Tuesday usually look like for this region")
historical_avg = (
    merged.groupby(['location', 'hour', 'day_of_week'])['demand_gw']
    .transform('mean')
)
merged['historical_avg_load'] = historical_avg

merged.to_parquet('/Users/shreya/Documents/College/Sem 5/ML/Lab/GridSense/data/processed/merged.parquet', index=False)
merged.head()

,datetime,location,demand_gw,temperature,humidity,hour,day_of_week,month,is_holiday,historical_avg_load
0,2019-01-01 00:00:00,National,118.69067,8.8,84,0,1,1,0,151.682120
1,2019-01-01 01:00:00,National,116.02923,8.1,86,1,1,1,0,148.307131
2,2019-01-01 02:00:00,National,114.04414,7.6,87,2,1,1,0,145.392680
3,2019-01-01 03:00:00,National,113.64897,7.3,88,3,1,1,0,143.746651
4,2019-01-01 04:00:00,National,116.29005,7.1,88,4,1,1,0,144.250696


In [23]:
print(merged['is_holiday'].sum(), "holiday-hours out of", len(merged))
print(merged[merged['is_holiday']==1]['datetime'].dt.date.unique()[:10])

13968 holiday-hours out of 280368
[datetime.date(2019, 1, 26) datetime.date(2019, 3, 4)
 datetime.date(2019, 3, 21) datetime.date(2019, 4, 13)
 datetime.date(2019, 4, 14) datetime.date(2019, 4, 17)
 datetime.date(2019, 4, 19) datetime.date(2019, 5, 18)
 datetime.date(2019, 6, 5) datetime.date(2019, 8, 12)]
